In [1]:
# Install required packages
!pip install transformers datasets seqeval evaluate accelerate -q

In [2]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Hugging Face
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
import evaluate

import torch

print(f"✅ PyTorch version : {torch.__version__}")
print(f"✅ CUDA available  : {torch.cuda.is_available()}")
print(f"✅ Pandas version  : {pd.__version__}")
print(f"✅ NumPy version   : {np.__version__}")

✅ PyTorch version : 2.11.0+cpu
✅ CUDA available  : False
✅ Pandas version  : 3.0.2
✅ NumPy version   : 2.4.4


In [ ]:

try:
    # Option A — cleanest, no loading script at all
    dataset = load_dataset("BramVanroy/conll2003")
    print("Loaded via BramVanroy/conll2003")
except Exception as e:
    print(f"Option A failed ({e}), trying Option B...")
    # Option B — load the parquet branch of eriktks/conll2003
    dataset = load_dataset("eriktks/conll2003", revision="convert/parquet")
    print("Loaded via eriktks/conll2003 (parquet revision)")

print("\n Dataset structure:")
print(dataset)

# Inspect a single sample
sample = dataset['train'][0]
print("\n Sample tokens :", sample['tokens'])
print(" POS tag IDs   :", sample['pos_tags'])
print(" Chunk tag IDs :", sample['chunk_tags'])


In [ ]:
pos_label_list   = dataset['train'].features['pos_tags'].feature.names
chunk_label_list = dataset['train'].features['chunk_tags'].feature.names

print(f"\n🔖 POS Labels ({len(pos_label_list)})   :", pos_label_list)
print(f"\n🔖 Chunk Labels ({len(chunk_label_list)}) :", chunk_label_list)

In [ ]:
# Pretty print sample with labels
df_sample = pd.DataFrame({
    'Token'    : sample['tokens'],
    'POS Tag'  : [pos_label_list[i] for i in sample['pos_tags']],
    'Chunk Tag': [chunk_label_list[i] for i in sample['chunk_tags']]
})
print("\n📋 Sample sentence with labels:")
print(df_sample.to_string(index=False))

In [6]:
# ─────────────────────────────────────────
# Task 2 ▸ Tokenizer Initialization
# ─────────────────────────────────────────
MODEL_CHECKPOINT = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

print(f"✅ Tokenizer loaded: {MODEL_CHECKPOINT}")

# Demonstrate subword tokenization
test_words = ["Washington", "works", "internationally"]
for word in test_words:
    tokens = tokenizer.tokenize(word)
    print(f"  '{word}' → {tokens}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Tokenizer loaded: distilbert-base-uncased
  'Washington' → ['washington']
  'works' → ['works']
  'internationally' → ['internationally']


In [ ]:
# ─────────────────────────────────────────
# Task 2 ▸ Label Alignment Function
# ─────────────────────────────────────────
def tokenize_and_align_labels(examples, label_column='pos_tags'):
    """
    Tokenizes input and aligns labels with subword tokens.
    
    Strategy:
      - First subword of a word  → real label
      - Subsequent subwords      → -100 (ignored in loss)
      - Special tokens [CLS/SEP] → -100
    """
    tokenized_inputs = tokenizer(
        examples['tokens'],
        truncation=True,
        is_split_into_words=True   # input is already word-tokenized
    )

    all_labels = []
    for i, label_ids in enumerate(examples[label_column]):
        word_ids     = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids_aligned = []

        for word_idx in word_ids:
            if word_idx is None:
                # Special token → ignore
                label_ids_aligned.append(-100)
            elif word_idx != previous_word_idx:
                # First subword → assign real label
                label_ids_aligned.append(label_ids[word_idx])
            else:
                # Continuation subword → ignore
                label_ids_aligned.append(-100)
            previous_word_idx = word_idx

        all_labels.append(label_ids_aligned)

    tokenized_inputs['labels'] = all_labels
    return tokenized_inputs


# ─── Apply preprocessing for POS Tagging ───
pos_tokenized_dataset = dataset.map(
    lambda x: tokenize_and_align_labels(x, label_column='pos_tags'),
    batched=True
)

# ─── Apply preprocessing for Chunking ───
chunk_tokenized_dataset = dataset.map(
    lambda x: tokenize_and_align_labels(x, label_column='chunk_tags'),
    batched=True
)

print("✅ Tokenization & label alignment complete!")
print("\n📐 POS tokenized sample keys:", list(pos_tokenized_dataset['train'][0].keys()))

In [ ]:
# Verify label alignment on one example
ex = pos_tokenized_dataset['train'][0]
tokens  = tokenizer.convert_ids_to_tokens(ex['input_ids'])
labels  = ex['labels']

align_df = pd.DataFrame({'Token': tokens, 'Label ID': labels})
align_df['Label'] = align_df['Label ID'].apply(
    lambda x: pos_label_list[x] if x != -100 else 'IGNORED'
)
print("\n🔍 Token-to-label alignment (first 20 tokens):")
print(align_df.head(20).to_string(index=False))

In [ ]:
# ─────────────────────────────────────────
# Task 3 ▸ Model Configuration Helper
# ─────────────────────────────────────────
def build_model(label_list):
    """
    Builds a DistilBERT token classification model.
    
    Args:
        label_list: list of label strings
    Returns:
        model, id2label, label2id
    """
    id2label  = {i: lbl for i, lbl in enumerate(label_list)}
    label2id  = {lbl: i for i, lbl in enumerate(label_list)}

    model = AutoModelForTokenClassification.from_pretrained(
        MODEL_CHECKPOINT,
        num_labels  = len(label_list),
        id2label    = id2label,
        label2id    = label2id,
        ignore_mismatched_sizes=True
    )
    return model, id2label, label2id


# Build POS model
pos_model, pos_id2label, pos_label2id = build_model(pos_label_list)
print(f"✅ POS Model ready — {len(pos_label_list)} labels")
print(f"   id2label sample: {dict(list(pos_id2label.items())[:5])}")

# Build Chunking model
chunk_model, chunk_id2label, chunk_label2id = build_model(chunk_label_list)
print(f"\n✅ Chunk Model ready — {len(chunk_label_list)} labels")
print(f"   id2label: {chunk_id2label}")

In [8]:
# ─────────────────────────────────────────
# Task 4 ▸ seqeval metric setup
# ─────────────────────────────────────────
seqeval_metric = evaluate.load("seqeval")

def make_compute_metrics(id2label):
    """
    Returns a compute_metrics function for the given id2label mapping.
    """
    def compute_metrics(eval_preds):
        logits, labels = eval_preds
        predictions    = np.argmax(logits, axis=-1)

        true_labels = [
            [id2label[l] for l in label_row if l != -100]
            for label_row in labels
        ]
        true_preds = [
            [id2label[p] for p, l in zip(pred_row, label_row) if l != -100]
            for pred_row, label_row in zip(predictions, labels)
        ]

        results = seqeval_metric.compute(
            predictions=true_preds,
            references =true_labels
        )
        return {
            'precision': round(results['overall_precision'], 4),
            'recall'   : round(results['overall_recall'],    4),
            'f1'       : round(results['overall_f1'],        4),
            'accuracy' : round(results['overall_accuracy'],  4),
        }
    return compute_metrics

In [ ]:
# ─────────────────────────────────────────
# Task 4 ▸ Training Helper Function
# ─────────────────────────────────────────
def train_model(model, tokenized_ds, id2label, output_dir, task_name):
    """
    Fine-tunes a token classification model.
    
    Args:
        model         : HuggingFace token classification model
        tokenized_ds  : preprocessed dataset
        id2label      : mapping from id to label string
        output_dir    : directory to save model
        task_name     : name for logging (POS / Chunking)
    Returns:
        trainer object
    """
    data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

    training_args = TrainingArguments(
        output_dir                  = output_dir,
        learning_rate               = 2e-5,
        per_device_train_batch_size = 16,
        per_device_eval_batch_size  = 16,
        num_train_epochs            = 3,
        weight_decay                = 0.01,
        eval_strategy               = "epoch",   # pandas 3.x compatible arg name
        save_strategy               = "epoch",
        load_best_model_at_end      = True,
        push_to_hub                 = False,
        logging_steps               = 100,
        report_to                   = "none",
    )

    trainer = Trainer(
        model            = model,
        args             = training_args,
        train_dataset    = tokenized_ds['train'],
        eval_dataset     = tokenized_ds['validation'],
        tokenizer        = tokenizer,
        data_collator    = data_collator,
        compute_metrics  = make_compute_metrics(id2label),
    )

    print(f"\n🚀 Starting training for {task_name}...")
    trainer.train()
    print(f"✅ {task_name} training complete!")
    return trainer


# ─── Train POS Tagging Model ───
pos_trainer = train_model(
    model         = pos_model,
    tokenized_ds  = pos_tokenized_dataset,
    id2label      = pos_id2label,
    output_dir    = "./pos_model",
    task_name     = "POS Tagging"
)

In [ ]:
# ─── Train Chunking Model ───
chunk_trainer = train_model(
    model         = chunk_model,
    tokenized_ds  = chunk_tokenized_dataset,
    id2label      = chunk_id2label,
    output_dir    = "./chunk_model",
    task_name     = "Chunking"
)

In [ ]:
# ─────────────────────────────────────────
# Task 5 ▸ Evaluate POS Model on Test Set
# ─────────────────────────────────────────
print("📊 Evaluating POS Tagging model on test set...")
pos_results = pos_trainer.evaluate(pos_tokenized_dataset['test'])

print("\n🏆 POS Tagging — Test Results:")
pos_eval_df = pd.DataFrame([
    {"Metric": "Precision", "Score": pos_results.get('eval_precision', 'N/A')},
    {"Metric": "Recall",    "Score": pos_results.get('eval_recall',    'N/A')},
    {"Metric": "F1 Score",  "Score": pos_results.get('eval_f1',        'N/A')},
    {"Metric": "Accuracy",  "Score": pos_results.get('eval_accuracy',  'N/A')},
])
print(pos_eval_df.to_string(index=False))

In [ ]:
# ─────────────────────────────────────────
# Task 5 ▸ Evaluate Chunking Model on Test Set
# ─────────────────────────────────────────
print("📊 Evaluating Chunking model on test set...")
chunk_results = chunk_trainer.evaluate(chunk_tokenized_dataset['test'])

print("\n🏆 Chunking — Test Results:")
chunk_eval_df = pd.DataFrame([
    {"Metric": "Precision", "Score": chunk_results.get('eval_precision', 'N/A')},
    {"Metric": "Recall",    "Score": chunk_results.get('eval_recall',    'N/A')},
    {"Metric": "F1 Score",  "Score": chunk_results.get('eval_f1',        'N/A')},
    {"Metric": "Accuracy",  "Score": chunk_results.get('eval_accuracy',  'N/A')},
])
print(chunk_eval_df.to_string(index=False))

In [ ]:
# ─────────────────────────────────────────
# Summary Comparison Table
# ─────────────────────────────────────────
summary_df = pd.DataFrame({
    'Metric'     : ['Precision', 'Recall', 'F1 Score', 'Accuracy'],
    'POS Tagging': [
        pos_results.get('eval_precision', 'N/A'),
        pos_results.get('eval_recall',    'N/A'),
        pos_results.get('eval_f1',        'N/A'),
        pos_results.get('eval_accuracy',  'N/A'),
    ],
    'Chunking'   : [
        chunk_results.get('eval_precision', 'N/A'),
        chunk_results.get('eval_recall',    'N/A'),
        chunk_results.get('eval_f1',        'N/A'),
        chunk_results.get('eval_accuracy',  'N/A'),
    ],
})
print("\n📊 Combined Evaluation Summary:")
print(summary_df.to_string(index=False))

In [ ]:
# ─────────────────────────────────────────
# Task 6 ▸ Inference Function
# ─────────────────────────────────────────
def predict_tags(sentence, model, id2label):
    """
    Predicts token-level labels for a given sentence.
    
    Args:
        sentence  : input string
        model     : fine-tuned token classification model
        id2label  : mapping from prediction id to label string
    Returns:
        list of (word, predicted_label) tuples
    """
    model.eval()
    words = sentence.split()

    # Tokenize with word_ids tracking
    encoding = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors='pt',
        truncation=True
    )

    with torch.no_grad():
        outputs = model(**encoding)

    # Get predicted label per token
    predictions = torch.argmax(outputs.logits, dim=-1)[0].tolist()
    word_ids    = encoding.word_ids(batch_index=0)

    # Map back: only keep prediction for first subword of each word
    word_predictions = {}
    for token_idx, word_idx in enumerate(word_ids):
        if word_idx is not None and word_idx not in word_predictions:
            word_predictions[word_idx] = id2label[predictions[token_idx]]

    return [(words[i], word_predictions[i]) for i in range(len(words))]


# ─── Test sentences ───
test_sentences = [
    "John works at Google in California",
    "The quick brown fox jumps over the lazy dog",
    "Barack Obama was the president of the United States",
]

for sent in test_sentences:
    pos_preds   = predict_tags(sent, pos_model,   pos_id2label)
    chunk_preds = predict_tags(sent, chunk_model, chunk_id2label)

    df_out = pd.DataFrame({
        'Word'      : [w for w, _ in pos_preds],
        'POS Tag'   : [t for _, t in pos_preds],
        'Chunk Tag' : [t for _, t in chunk_preds],
    })
    print(f"\n🔤 Input : {sent}")
    print(df_out.to_string(index=False))
    print("-" * 50)

In [ ]:
# ─────────────────────────────────────────
# Task 7 ▸ Side-by-side visual comparison
# ─────────────────────────────────────────
comparison_sentence = "John works at Google in California"

pos_preds   = predict_tags(comparison_sentence, pos_model,   pos_id2label)
chunk_preds = predict_tags(comparison_sentence, chunk_model, chunk_id2label)

comp_df = pd.DataFrame({
    'Word'       : [w for w, _ in pos_preds],
    'POS Tag'    : [t for _, t in pos_preds],
    'Chunk Tag'  : [t for _, t in chunk_preds],
    'POS Meaning': [
        'Proper noun', 'Verb 3rd person', 'Preposition',
        'Proper noun', 'Preposition', 'Proper noun'
    ],
    'Chunk Meaning': [
        'Begin Noun Phrase', 'Begin Verb Phrase', 'Begin Prep Phrase',
        'Begin Noun Phrase', 'Begin Prep Phrase', 'Begin Noun Phrase'
    ],
})

print("\n🔍 Detailed Comparison:")
print(f"   Sentence: '{comparison_sentence}'")
print()
print(comp_df.to_string(index=False))

print("\n📌 Key Observations:")
print("  • POS tagging assigns a grammatical role to EACH word independently")
print("  • Chunking groups words into meaningful PHRASES using BIO notation")
print("  • 'John' → POS: NNP (proper noun) | Chunk: B-NP (start of noun phrase)")
print("  • Chunking requires understanding word sequences, making it harder")

In [ ]:
# ─────────────────────────────────────────
# Performance Comparison Summary
# ─────────────────────────────────────────
perf_df = pd.DataFrame({
    'Task'            : ['POS Tagging', 'Chunking'],
    'Difficulty'      : ['⭐ Easier', '⭐⭐ Harder'],
    'Label Schema'    : ['Fine-grained (40+ tags)', 'BIO format (phrase-level)'],
    'Expected F1'     : ['~95%', '~91-93%'],
    'Main Challenge'  : ['Ambiguous words', 'Span boundary detection'],
})
print("\n📊 Task Comparison Summary:")
print(perf_df.to_string(index=False))